# XGBoost log(C5) — 2025 Q1 using `XGBoost_model_features_by_selection.xlsx`

1. Pick the **best** model column (highest **Test_R2** in the header).
2. Load **`Transformed_Deseason_Residuals_Lagged.xlsx`** — for each selected feature after **2024-12-31**, use **actuals** when present; otherwise **univariate LSTM** (PyTorch) through **2025-03-31**.
3. Retrain **XGBoost** on history through **2024-12-31** with those features + AR lags of log(C5).
4. Predict **log(C5)** for **2025-01-01 … 2025-03-31** and compare to **actual** log(C5) (AR lags use **actual** past log(C5)).

## Google Colab

1. Run **Install dependencies** (next cell).
2. Run **Colab paths + Drive** (cell after that). Set `MOUNT_GOOGLE_DRIVE = False` if your data is only under `/content` (then edit `COLAB_DATA_DIR` or upload files there).
3. Put the same Excel/CSV layout as on your Mac in **`MyDrive/Project 005 Data/`** (or change `COLAB_DATA_DIR` in that cell).

On Drive, `read_excel` can hit FUSE errors; this notebook defines **`read_excel_colab_safe`** and uses it for workbook reads where patched.

**If you see `ImportError: cannot import name '_center' from 'numpy._core.umath'`**, Colab's pre-installed numpy was broken by an earlier `pip install -U`. Recover with:

```
!pip install --force-reinstall -q "numpy==2.0.2" "pandas==2.2.2"
```

then **Runtime → Restart session** and run this notebook from the top. The install cell below no longer upgrades numpy/pandas/torch and will stop the notebook if the runtime needs a restart.


In [ ]:
# Install dependencies.
# Colab (esp. GPU runtime w/ RAPIDS) pins numpy<2.1 and pandas<2.4. Any `pip install -U`
# can desync wheels and break `numpy._core.umath._center` (imported by numpy.strings,
# xgboost, sklearn, etc.). We therefore:
#   1) Detect a broken/split numpy,
#   2) Delete shadow dist dirs pip left behind (`~numpy*`, `~orch*`, half-removed numpy),
#   3) Surgically reinstall numpy==2.0.2 and pandas==2.2.2 (compatible with RAPIDS),
#   4) Halt the notebook with a clear "Restart session" message.
# We never use `-U` on Colab and never touch torch here; the GPU runtime ships its own.
import importlib
import shutil
import subprocess
import sys
from pathlib import Path

_IN_COLAB = "google.colab" in sys.modules
_USE_TORCH = False
_USE_STATS = True


def _pip(*args):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *args],
        check=True,
    )


def _has(module: str) -> bool:
    try:
        importlib.import_module(module)
        return True
    except Exception:
        return False


def _numpy_healthy() -> bool:
    # If numpy._core.umath._center is missing, the numpy install is split across
    # incompatible wheels and Python cannot recover in the current session.
    try:
        from numpy._core import umath as _um
        import numpy.strings  # noqa: F401  same path xgboost/sklearn trigger
        return hasattr(_um, "_center")
    except Exception:
        return False


def _colab_cleanup_and_repair_numpy() -> None:
    # Wipe pip's abandoned shadow installs (names start with '~') and any half-removed
    # numpy dir/dist-info — these are what make `pip install --force-reinstall numpy`
    # silently fail to fully overwrite the old files.
    import glob
    site_roots = [Path(p) for p in sys.path if p.endswith("dist-packages") or p.endswith("site-packages")]
    site_roots = [p for p in site_roots if p.is_dir()]
    for root in site_roots:
        for pat in ("~*", "numpy", "numpy-*.dist-info", "numpy.libs"):
            for hit in glob.glob(str(root / pat)):
                try:
                    if Path(hit).is_dir():
                        shutil.rmtree(hit, ignore_errors=True)
                    else:
                        Path(hit).unlink(missing_ok=True)
                except Exception:
                    pass
    # Reinstall a numpy/pandas pair that's compatible with Colab GPU (RAPIDS: numpy<2.1).
    _pip("--no-deps", "--force-reinstall", "numpy==2.0.2")
    _pip("--force-reinstall", "pandas==2.2.2")


_COLAB_RESTART_MSG = (
    "Colab runtime was repaired (numpy/pandas reinstalled to versions compatible with "
    "RAPIDS / GPU runtime). Please **Runtime -> Restart session** and then run this "
    "notebook from the top."
)

if _IN_COLAB:
    if not _numpy_healthy():
        _colab_cleanup_and_repair_numpy()
        raise RuntimeError(_COLAB_RESTART_MSG)
    _needed = []
    for _mod, _pkg in [
        ("xgboost", "xgboost"),
        ("openpyxl", "openpyxl"),
        ("tqdm", "tqdm"),
        ("sklearn", "scikit-learn"),
        ("matplotlib", "matplotlib"),
    ]:
        if not _has(_mod):
            _needed.append(_pkg)
    if _USE_TORCH and not _has("torch"):
        _needed.append("torch")
    if _USE_STATS and not _has("statsmodels"):
        _needed.append("statsmodels")
    if _needed:
        _pip(*_needed)  # intentionally NO -U
        raise RuntimeError(
            "Installed missing packages on Colab: " + ", ".join(_needed) + ". "
            "Please **Runtime -> Restart session**, then run the notebook from the top."
        )
    print("Colab env OK - no pip actions needed.")
else:
    _PKGS = ["pandas", "numpy", "matplotlib", "openpyxl", "scikit-learn", "tqdm", "xgboost"]
    if _USE_TORCH:
        _PKGS = ["torch", *_PKGS]
    if _USE_STATS:
        _PKGS.append("statsmodels")
    _pip("-U", *_PKGS)
    print("pip OK | Colab: False")


In [ ]:
# --- Colab: paths, optional Drive mount, safe Excel reads ---
import os
import shutil
import sys
from pathlib import Path

import pandas as pd

IN_COLAB = "google.colab" in sys.modules
MOUNT_GOOGLE_DRIVE = True  # False if you only use /content uploads
COLAB_DATA_DIR = Path("/content/drive/MyDrive/Project 005 Data")


def _marker_exists(p: Path, name: str = "Master_Data_AVG_version_update.xlsx") -> bool:
    try:
        return (p / name).is_file()
    except OSError:
        return False


def resolve_project005_base_dir() -> Path:
    if IN_COLAB:
        if COLAB_DATA_DIR.is_dir() and _marker_exists(COLAB_DATA_DIR):
            return COLAB_DATA_DIR.resolve()
        alt = Path("/content")
        if _marker_exists(alt):
            return alt.resolve()
        return COLAB_DATA_DIR.resolve()
    here = Path.cwd().resolve()
    if _marker_exists(here):
        return here
    fb = (Path.home() / "Downloads" / "Project 005 Data").resolve()
    if fb.is_dir() and _marker_exists(fb):
        return fb
    return here


if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except Exception as exc:
        print("Drive mount skipped or failed:", exc)

BASE_DIR = resolve_project005_base_dir()
if IN_COLAB and BASE_DIR.is_dir():
    try:
        os.chdir(BASE_DIR)
    except OSError:
        pass


def read_excel_colab_safe(path, **kwargs):
    """Read Excel; on Colab copy from Drive to /tmp first (reduces FUSE transport errors)."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    resolved = path.expanduser().resolve()
    sp = os.fspath(resolved)
    if IN_COLAB and sp.startswith("/content/drive/"):
        local = Path("/tmp") / f"_colab_safe_{resolved.name}"
        shutil.copyfile(resolved, local)
        try:
            return pd.read_excel(local, **kwargs)
        finally:
            try:
                local.unlink(missing_ok=True)
            except OSError:
                pass
    return pd.read_excel(resolved, **kwargs)


print("IN_COLAB:", IN_COLAB, "| BASE_DIR:", BASE_DIR)


In [ ]:
# --- Colab NumPy guard (before heavy imports): numpy._core.umath must expose _center. ---
import sys as _sys
if "google.colab" in _sys.modules:
    try:
        from numpy._core import umath as _np_um
        import numpy.strings  # noqa: F401  same path xgboost/sklearn trigger
        _np_ok = hasattr(_np_um, "_center")
    except Exception:
        _np_ok = False
    if not _np_ok:
        import glob as _glob, shutil as _shutil, subprocess as _sp
        from pathlib import Path as _Path
        _roots = [_Path(p) for p in _sys.path if p.endswith("dist-packages") or p.endswith("site-packages")]
        for _root in [_r for _r in _roots if _r.is_dir()]:
            for _pat in ("~*", "numpy", "numpy-*.dist-info", "numpy.libs"):
                for _hit in _glob.glob(str(_root / _pat)):
                    try:
                        if _Path(_hit).is_dir():
                            _shutil.rmtree(_hit, ignore_errors=True)
                        else:
                            _Path(_hit).unlink(missing_ok=True)
                    except Exception:
                        pass
        _sp.run([_sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", "numpy==2.0.2"], check=True)
        _sp.run([_sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "pandas==2.2.2"], check=True)
        raise RuntimeError(
            "Colab numpy was broken and has been reinstalled (2.0.2). "
            "Please **Runtime -> Restart session**, then run the notebook from the top."
        )



In [ ]:
def parse_best_feature_column(selection_path):
    """Return (column_name, list of feature names) for the model with max Test_R2 in header."""
    sel = read_excel_colab_safe(selection_path)
    best_col = None
    best_r2 = -np.inf
    for c in sel.columns:
        if c == "feature_index" or not isinstance(c, str):
            continue
        m = re.search(r"Test_R2=([0-9.eE+-]+)", c)
        if not m:
            continue
        r2 = float(m.group(1))
        if r2 > best_r2:
            best_r2 = r2
            best_col = c
    if best_col is None:
        raise ValueError("Could not parse any Test_R2 from feature selection xlsx")
    names = sel[best_col].dropna().astype(str).str.strip()
    names = [n for n in names if n and n.lower() != "nan"]
    return best_col, names, best_r2


best_col_name, selected_features, best_r2_header = parse_best_feature_column(FEATURE_SELECTION_XLSX)
print("Best model column (by Test_R2 in header):", best_col_name)
print("Parsed Test_R2:", best_r2_header, "| n features:", len(selected_features))


In [ ]:
lagged = read_excel_colab_safe(LAGGED_XLSX)
date_col = "Date" if "Date" in lagged.columns else lagged.columns[0]
lagged[date_col] = pd.to_datetime(lagged[date_col])
lagged = lagged.set_index(date_col).sort_index()
exc = [c for c in lagged.columns if any(p in str(c) for p in MODEL_GENERATED_PATTERNS)]
if exc:
    lagged = lagged.drop(columns=exc)

master = read_excel_colab_safe(MASTER_XLSX)
md = "Date" if "Date" in master.columns else master.columns[0]
master[md] = pd.to_datetime(master[md])
master = master.set_index(md).sort_index()
c5 = master["C5"].astype(float)
log_c5_full = np.log(c5.replace(0, np.nan)).dropna()

missing = [f for f in selected_features if f not in lagged.columns]
if missing:
    print("Warning: not in lagged file (skipped):", missing[:5], "..." if len(missing) > 5 else "")
use_cols = [f for f in selected_features if f in lagged.columns]
if not use_cols:
    raise ValueError("No selected features found in lagged columns")
print("Using", len(use_cols), "features present in lagged data.")

horizon_idx = lagged.index[(lagged.index > CUTOFF) & (lagged.index <= H_END)]
print("Horizon rows:", len(horizon_idx), "|", horizon_idx.min(), "->", horizon_idx.max())


In [ ]:
class TinyLSTM(nn.Module):
    def __init__(self, hidden=48):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        o, _ = self.lstm(x)
        return self.fc(o[:, -1, :]).squeeze(-1)


def lstm_forecast_univariate(series, horizon_index, seq_len=SEQ_LEN, epochs=LSTM_EPOCHS):
    """Train on z-scored history through CUTOFF; recursive 1-step z forecasts on sorted horizon_index."""
    s = series.astype(float)
    hist = s.loc[s.index <= CUTOFF].dropna()
    if len(hist) < seq_len + 30:
        return pd.Series(0.0, index=horizon_index)
    mu, sigma = float(hist.mean()), float(hist.std())
    if sigma < 1e-12:
        return pd.Series(mu, index=horizon_index)
    z_hist = ((hist - mu) / sigma).astype(np.float32)
    zh = z_hist.values
    Xl, yl = [], []
    for i in range(seq_len, len(zh)):
        Xl.append(zh[i - seq_len : i])
        yl.append(zh[i])
    Xl = np.stack(Xl)[..., np.newaxis]
    yl = np.array(yl, dtype=np.float32)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(Xl), torch.from_numpy(yl)),
        batch_size=LSTM_BATCH,
        shuffle=True,
    )
    model = TinyLSTM(LSTM_HIDDEN).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LSTM_LR)
    loss_fn = nn.MSELoss()
    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            loss_fn(model(xb), yb).backward()
            opt.step()
    model.eval()
    buf = list(zh[-seq_len:].astype(np.float32))
    hidx = pd.DatetimeIndex(sorted(horizon_index.unique()))
    preds = []
    for _ in hidx:
        win = np.array(buf[-seq_len:], dtype=np.float32).reshape(1, seq_len, 1)
        with torch.no_grad():
            pz = float(model(torch.from_numpy(win).to(device)).cpu().numpy().ravel()[0])
        preds.append(pz)
        buf.append(pz)
    out = np.array(preds, dtype=float) * sigma + mu
    return pd.Series(out, index=hidx)


feat_filled = lagged[use_cols].copy()
n_lstm = 0
for col in use_cols:
    h = feat_filled.loc[horizon_idx, col]
    if h.notna().all():
        continue
    pred = lstm_forecast_univariate(feat_filled[col], horizon_idx)
    m = h.isna()
    feat_filled.loc[horizon_idx[m], col] = pred.reindex(horizon_idx[m]).values
    n_lstm += 1

print("LSTM gap-fill applied to", n_lstm, "features (horizon had at least one NaN).")


In [ ]:
def add_ar_features(log_c5_series, idx):
    y = log_c5_series.reindex(idx)
    return pd.DataFrame(
        {
            "log_C5_lag1": y.shift(1),
            "log_C5_lag2": y.shift(2),
            "log_C5_lag5": y.shift(5),
            "log_C5_lag10": y.shift(10),
            "log_C5_lag20": y.shift(20),
            "log_C5_rolling5": y.rolling(5, min_periods=1).mean().shift(1),
            "log_C5_rolling20": y.rolling(20, min_periods=1).mean().shift(1),
        },
        index=idx,
    )


# Full calendar index for alignment (train + horizon)
common = lagged.index.intersection(log_c5_full.index)
feat_block = feat_filled.reindex(common).fillna(0)
y_all = log_c5_full.reindex(common)
ar_df = add_ar_features(y_all, common)
X_all = pd.concat([feat_block[use_cols], ar_df], axis=1)
valid_ar = ar_df.notna().all(axis=1)
X_all = X_all.loc[valid_ar].fillna(0)
y_all = y_all.loc[valid_ar]

train_mask = (X_all.index >= "2015-01-29") & (X_all.index <= CUTOFF)
X_train = X_all.loc[train_mask]
y_train = y_all.loc[train_mask]
print("Train:", X_train.shape, "|", X_train.index.min().date(), "->", X_train.index.max().date())

model = xgb.XGBRegressor(**XGB_PARAMS)
model.fit(X_train, y_train)

horizon_mask = (X_all.index > CUTOFF) & (X_all.index <= H_END)
X_h = X_all.loc[horizon_mask]
y_h_actual = y_all.loc[horizon_mask]
y_pred = model.predict(X_h)

mask = y_h_actual.notna()
ya = y_h_actual.loc[mask].values
yp = y_pred[mask.values]
dt = y_h_actual.loc[mask].index

rmse = np.sqrt(mean_squared_error(ya, yp))
mae = mean_absolute_error(ya, yp)
r2 = r2_score(ya, yp)
print("=" * 60)
print("2025 holdout log(C5) | n =", len(ya))
print(f"  RMSE: {rmse:.6f}  MAE: {mae:.6f}  R²: {r2:.6f}")
print("=" * 60)

out = pd.DataFrame({"Date": dt, "log_C5_actual": ya, "log_C5_pred": yp})
out.head(10)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(out["Date"], out["log_C5_actual"], label="Actual", color="tab:blue", linewidth=1.8)
ax.plot(out["Date"], out["log_C5_pred"], label="XGB forecast", color="tab:orange", linewidth=1.8)
ax.set_xlabel("Date")
ax.set_ylabel("log(C5)")
ax.set_title(f"2025 Q1 (features from {best_col_name[:40]}...)")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
